# 02 Prepare FinanceBench Dataset

Σε αυτό το notebook προετοιμάζεται το αρχικό δείγμα του FinanceBench για χρήση στα επόμενα στάδια. Γίνεται φόρτωση των ερωτήσεων και των διαθέσιμων εγγράφων, έλεγχος βασικών πεδίων και δημιουργία του working dataset που χρησιμοποιείται στο parsing και στην αξιολόγηση.


In [ ]:

from pathlib import Path
import base64
import json
import os
import re
import shutil
import time
import urllib.parse
import urllib.request
import warnings

import pandas as pd


In [ ]:

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PDFS_DIR = RAW_DIR / "pdfs"
INTERIM_DIR = DATA_DIR / "interim"

for path in [DATA_DIR, RAW_DIR, PDFS_DIR, INTERIM_DIR]:
    path.mkdir(parents=True, exist_ok=True)

QUESTIONS_PATH = RAW_DIR / "financebench_open_source.jsonl"
DOC_INFO_PATH = RAW_DIR / "financebench_document_information.jsonl"

HF_DATASET_NAME = "PatronusAI/financebench"
DOWNLOAD_SOURCE_DATA_IF_MISSING = True
DOWNLOAD_PDFS_IF_MISSING = True
PDF_DOWNLOAD_TIMEOUT_SECONDS = 120
PDF_DOWNLOAD_SLEEP_SECONDS = 0.2

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("PDFS_DIR:", PDFS_DIR)
print("QUESTIONS_PATH exists:", QUESTIONS_PATH.exists())
print("DOC_INFO_PATH exists:", DOC_INFO_PATH.exists())


In [ ]:
KAGGLE_DATASET_ENV = "FINANCEBENCH_KAGGLE_DATASET_DIR"
dataset_dir_value = os.getenv(KAGGLE_DATASET_ENV)
KAGGLE_DATASET_DIR = (
    Path(dataset_dir_value).expanduser().resolve()
    if dataset_dir_value
    else None
)
REQUIRE_KAGGLE_DATASET = os.getenv("FINANCEBENCH_REQUIRE_KAGGLE_DATASET") == "1"


def read_attached_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if path.suffix.lower() == ".jsonl":
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        try:
            return pd.read_json(path, lines=True)
        except ValueError:
            return pd.read_json(path)
    raise ValueError(f"Unsupported table format: {path}")


def normalize_attached_financebench(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    if "evidence" not in frame.columns and "evidence_text" in frame.columns:
        def evidence_record(row):
            page_number = row.get("page_number")
            if pd.isna(page_number):
                page_number = None
            return [{
                "doc_name": row.get("doc_name"),
                "evidence_page_num": page_number,
                "evidence_text": row.get("evidence_text", ""),
            }]
        frame["evidence"] = frame.apply(evidence_record, axis=1)

    defaults = {
        "company": frame["doc_name"].astype(str).str.split("_").str[0],
        "doc_type": frame["doc_name"].astype(str).str.split("_").str[-1],
        "doc_period": None,
        "question_type": None,
        "question_reasoning": None,
        "domain_question_num": None,
        "justification": None,
        "dataset_subset_label": "OPEN_SOURCE",
        "gics_sector": None,
        "doc_link": None,
        "evidence": None,
    }
    for column, default in defaults.items():
        if column not in frame.columns:
            frame[column] = default
    return frame


attached_table_path = None
if KAGGLE_DATASET_DIR and KAGGLE_DATASET_DIR.is_dir():
    table_paths = sorted(
        path
        for path in KAGGLE_DATASET_DIR.rglob("*")
        if path.is_file() and path.suffix.lower() in {".csv", ".parquet", ".pq", ".jsonl", ".json"}
    )
    required_columns = {"financebench_id", "question", "answer", "doc_name"}
    attached_frame = None
    table_errors = []
    for table_path in table_paths:
        try:
            candidate_frame = read_attached_table(table_path)
        except Exception as exc:
            table_errors.append(f"{table_path.name}: {type(exc).__name__}: {exc}")
            continue
        if required_columns.issubset(candidate_frame.columns):
            attached_frame = normalize_attached_financebench(candidate_frame)
            attached_table_path = table_path
            break

    if attached_frame is not None:
        question_columns = [
            "financebench_id", "company", "doc_name", "question_type",
            "question_reasoning", "domain_question_num", "question", "answer",
            "justification", "dataset_subset_label", "evidence",
        ]
        document_columns = [
            "doc_name", "company", "gics_sector", "doc_type",
            "doc_period", "doc_link",
        ]
        attached_frame[question_columns].to_json(
            QUESTIONS_PATH, orient="records", lines=True, force_ascii=False
        )
        (
            attached_frame[document_columns]
            .drop_duplicates(subset=["doc_name"])
            .sort_values("doc_name")
            .to_json(DOC_INFO_PATH, orient="records", lines=True, force_ascii=False)
        )
        print(f"Bootstrapped FinanceBench tables from: {attached_table_path}")
    elif REQUIRE_KAGGLE_DATASET:
        details = "; ".join(table_errors[:5])
        raise RuntimeError(
            f"No compatible FinanceBench table found under {KAGGLE_DATASET_DIR}. "
            f"Expected columns: {sorted(required_columns)}. {details}"
        )

    attached_pdfs = sorted(KAGGLE_DATASET_DIR.rglob("*.pdf"))
    duplicate_names = {
        path.name
        for path in attached_pdfs
        if sum(other.name == path.name for other in attached_pdfs) > 1
    }
    if duplicate_names:
        raise RuntimeError(f"Duplicate PDF filenames in Kaggle dataset: {sorted(duplicate_names)}")

    linked_pdfs = 0
    for source_pdf in attached_pdfs:
        destination = PDFS_DIR / source_pdf.name
        if destination.exists():
            continue
        try:
            destination.symlink_to(source_pdf)
        except OSError:
            shutil.copy2(source_pdf, destination)
        linked_pdfs += 1
    print(f"Attached Kaggle PDFs available locally: {len(attached_pdfs)} ({linked_pdfs} added)")
else:
    print("No attached Kaggle dataset resolved; Hugging Face and document URLs remain the fallback.")


In [ ]:

def write_jsonl(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(path, orient="records", lines=True, force_ascii=False)


def load_financebench_from_huggingface() -> pd.DataFrame:
    try:
        from datasets import load_dataset
    except ImportError as exc:
        raise ImportError(
            "Missing dependency 'datasets'. Install requirements.txt before running "
            "the notebook from an empty data folder."
        ) from exc

    print(f"Downloading/loading {HF_DATASET_NAME} from Hugging Face...")
    dataset = load_dataset(HF_DATASET_NAME, split="train")
    df = dataset.to_pandas()
    print("Loaded Hugging Face dataset:", df.shape)
    return df


def ensure_source_jsonl_files() -> None:
    if QUESTIONS_PATH.exists() and DOC_INFO_PATH.exists():
        print("Raw FinanceBench JSONL files already exist.")
        return

    if not DOWNLOAD_SOURCE_DATA_IF_MISSING:
        missing = [str(p) for p in [QUESTIONS_PATH, DOC_INFO_PATH] if not p.exists()]
        raise FileNotFoundError(
            "Missing raw FinanceBench files and DOWNLOAD_SOURCE_DATA_IF_MISSING=False: "
            + ", ".join(missing)
        )

    hf_df = load_financebench_from_huggingface()

    question_cols = [
        "financebench_id",
        "company",
        "doc_name",
        "question_type",
        "question_reasoning",
        "domain_question_num",
        "question",
        "answer",
        "justification",
        "dataset_subset_label",
        "evidence",
    ]
    question_cols = [c for c in question_cols if c in hf_df.columns]

    doc_cols = [
        "doc_name",
        "company",
        "gics_sector",
        "doc_type",
        "doc_period",
        "doc_link",
    ]
    doc_cols = [c for c in doc_cols if c in hf_df.columns]

    df_questions_to_save = hf_df[question_cols].copy()
    df_docs_to_save = (
        hf_df[doc_cols]
        .drop_duplicates(subset=["doc_name"])
        .sort_values("doc_name")
        .reset_index(drop=True)
    )

    write_jsonl(df_questions_to_save, QUESTIONS_PATH)
    write_jsonl(df_docs_to_save, DOC_INFO_PATH)

    print("Created raw FinanceBench JSONL files:")
    print("-", QUESTIONS_PATH)
    print("-", DOC_INFO_PATH)


def pdf_filename_for_doc(doc_name: str) -> str:
    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(doc_name).strip()).strip("._")
    if not stem:
        stem = "financebench_document"
    return f"{stem}.pdf"


def resolve_pdf_url(url: str) -> str:
    """Resolve Adobe pdf-page redirect links used by a few FinanceBench rows."""
    parsed = urllib.parse.urlparse(str(url))
    query = urllib.parse.parse_qs(parsed.query)
    pdf_target = query.get("pdfTarget", [None])[0]
    if pdf_target:
        try:
            return base64.b64decode(pdf_target).decode("utf-8")
        except Exception:
            return str(url)
    return str(url)


def download_file(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    resolved_url = resolve_pdf_url(url)
    request = urllib.request.Request(
        resolved_url,
        headers={"User-Agent": "Mozilla/5.0 (FinanceBench reproducibility script)"},
    )

    tmp_path = destination.with_suffix(destination.suffix + ".part")
    with urllib.request.urlopen(request, timeout=PDF_DOWNLOAD_TIMEOUT_SECONDS) as response:
        with open(tmp_path, "wb") as f:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)

    if tmp_path.stat().st_size == 0:
        tmp_path.unlink(missing_ok=True)
        raise ValueError(f"Downloaded empty file from {resolved_url}")

    tmp_path.replace(destination)


def ensure_financebench_pdfs(df_docs: pd.DataFrame) -> pd.DataFrame:
    if "doc_link" not in df_docs.columns:
        print("No doc_link column found; skipping PDF download.")
        return pd.DataFrame()

    docs = (
        df_docs[["doc_name", "doc_link"]]
        .dropna(subset=["doc_name", "doc_link"])
        .drop_duplicates(subset=["doc_name"])
        .copy()
    )
    docs["pdf_filename"] = docs["doc_name"].apply(pdf_filename_for_doc)
    docs["pdf_path"] = docs["pdf_filename"].apply(lambda name: PDFS_DIR / name)
    missing_docs = docs[~docs["pdf_path"].apply(lambda p: p.exists())].copy()

    print("FinanceBench unique docs:", len(docs))
    print("Existing PDFs:", len(docs) - len(missing_docs))
    print("Missing PDFs:", len(missing_docs))

    if missing_docs.empty:
        return docs

    if not DOWNLOAD_PDFS_IF_MISSING:
        raise FileNotFoundError(
            f"{len(missing_docs)} PDFs are missing and DOWNLOAD_PDFS_IF_MISSING=False."
        )

    failures = []
    for _, row in missing_docs.iterrows():
        destination = Path(row["pdf_path"])
        try:
            print(f"Downloading {row['doc_name']} -> {destination.name}")
            download_file(row["doc_link"], destination)
            time.sleep(PDF_DOWNLOAD_SLEEP_SECONDS)
        except Exception as exc:
            failures.append({
                "doc_name": row["doc_name"],
                "doc_link": row["doc_link"],
                "error": f"{type(exc).__name__}: {exc}",
            })
            print(f"Failed: {row['doc_name']} ({type(exc).__name__}: {exc})")

    if failures:
        failure_df = pd.DataFrame(failures)
        failure_path = INTERIM_DIR / "financebench_pdf_download_failures.csv"
        failure_df.to_csv(failure_path, index=False)
        raise RuntimeError(
            f"Failed to download {len(failures)} FinanceBench PDFs. "
            f"Details saved to {failure_path}."
        )

    return docs


ensure_source_jsonl_files()

df_questions = pd.read_json(QUESTIONS_PATH, lines=True)
df_docs = pd.read_json(DOC_INFO_PATH, lines=True)

pdf_download_manifest_df = ensure_financebench_pdfs(df_docs)
if len(pdf_download_manifest_df):
    pdf_download_manifest_df.assign(
        pdf_path=pdf_download_manifest_df["pdf_path"].astype(str)
    ).to_csv(INTERIM_DIR / "financebench_pdf_download_manifest.csv", index=False)

print("df_questions shape:", df_questions.shape)
print("df_docs shape:", df_docs.shape)


In [ ]:
print("Questions columns:")
for col in df_questions.columns:
    print("-", col)

print("\nDocument info columns:")
for col in df_docs.columns:
    print("-", col)

In [ ]:
display(df_questions.head(2))
display(df_docs.head(2))

In [ ]:
df_full = pd.merge(
    df_questions,
    df_docs,
    on="doc_name",
    how="left",
    suffixes=("", "_doc")
)

print("df_full shape:", df_full.shape)
df_full.head(3)

In [ ]:
merge_summary = {
    "question_rows": len(df_questions),
    "doc_rows": len(df_docs),
    "merged_rows": len(df_full),
    "missing_doc_metadata_rows": int(df_full["doc_type"].isna().sum()) if "doc_type" in df_full.columns else None,
    "unique_doc_names_questions": df_questions["doc_name"].nunique(),
    "unique_doc_names_docs": df_docs["doc_name"].nunique(),
}

pd.DataFrame([merge_summary])

In [ ]:
missing_df = pd.DataFrame({
    "column": df_full.columns,
    "missing_count": df_full.isna().sum().values,
    "missing_pct": (df_full.isna().mean().values * 100).round(2)
}).sort_values("missing_pct", ascending=False)

missing_df

In [ ]:
record = df_full.iloc[0].to_dict()

for k, v in record.items():
    print(f"{k}: {v}\n")

In [ ]:
print(type(df_full.loc[0, "evidence"]))
print(df_full.loc[0, "evidence"])

In [ ]:
first_evidence = df_full.loc[0, "evidence"][0] if len(df_full.loc[0, "evidence"]) > 0 else {}
first_evidence

In [ ]:
stats = {
    "n_questions": len(df_full),
    "n_unique_companies": df_full["company"].nunique(),
    "n_unique_docs": df_full["doc_name"].nunique(),
    "question_types": df_full["question_type"].nunique() if "question_type" in df_full.columns else None,
    "reasoning_types": df_full["question_reasoning"].nunique() if "question_reasoning" in df_full.columns else None,
}

pd.DataFrame([stats])

In [ ]:
if "question_type" in df_full.columns:
    display(df_full["question_type"].value_counts(dropna=False).to_frame("count"))

if "question_reasoning" in df_full.columns:
    display(df_full["question_reasoning"].value_counts(dropna=False).head(20).to_frame("count"))

if "doc_type" in df_full.columns:
    display(df_full["doc_type"].value_counts(dropna=False).to_frame("count"))

In [ ]:
pdf_files = sorted(PDFS_DIR.glob("*.pdf"))

pdf_inventory = pd.DataFrame({
    "pdf_filename": [p.name for p in pdf_files],
    "pdf_stem": [p.stem for p in pdf_files],
    "pdf_path": [str(p) for p in pdf_files]
})

print("Local PDFs found:", len(pdf_inventory))
pdf_inventory.head()

In [ ]:
def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = s.replace(".pdf", "")
    s = s.replace("_", " ")
    s = s.replace("-", " ")
    s = " ".join(s.split())
    return s

df_full["normalized_doc_name"] = df_full["doc_name"].apply(normalize_text)
pdf_inventory["normalized_pdf_stem"] = pdf_inventory["pdf_stem"].apply(normalize_text)

display(df_full[["doc_name", "normalized_doc_name"]].head())
display(pdf_inventory.head())

In [ ]:
def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = s.replace(".pdf", "")
    s = s.replace("_", " ")
    s = s.replace("-", " ")
    s = " ".join(s.split())
    return s

In [ ]:
df_matched = df_full.merge(
    pdf_inventory,
    left_on="normalized_doc_name",
    right_on="normalized_pdf_stem",
    how="left"
)

match_summary = {
    "total_rows": len(df_matched),
    "matched_rows": int(df_matched["pdf_filename"].notna().sum()),
    "unmatched_rows": int(df_matched["pdf_filename"].isna().sum()),
    "unique_docs_in_dataset": df_matched["doc_name"].nunique(),
    "unique_local_pdfs": len(pdf_inventory),
    "matched_unique_docs": df_matched.loc[df_matched["pdf_filename"].notna(), "doc_name"].nunique()
}

pd.DataFrame([match_summary])

In [ ]:
unmatched_docs = (
    df_matched.loc[df_matched["pdf_filename"].isna(), ["doc_name", "company", "doc_type", "doc_period"]]
    .drop_duplicates()
    .sort_values(["company", "doc_name"])
)

print("Unmatched unique docs:", len(unmatched_docs))
unmatched_docs.head(20)

In [ ]:
company_counts = (
    df_matched["company"]
    .value_counts()
    .reset_index()
)
company_counts.columns = ["company", "question_count"]

doc_counts = (
    df_matched["doc_name"]
    .value_counts()
    .reset_index()
)
doc_counts.columns = ["doc_name", "question_count"]

display(company_counts.head(15))
display(doc_counts.head(15))

In [ ]:
for col in ["question", "answer", "justification"]:
    if col in df_matched.columns:
        lengths = df_matched[col].fillna("").astype(str).str.len()
        print(f"\nColumn: {col}")
        print(lengths.describe())

In [ ]:
working_df = df_matched.copy().reset_index(drop=True)
working_df["row_id"] = working_df.index

priority_cols = [
    "row_id",
    "financebench_id",
    "question",
    "answer",
    "company",
    "doc_name",
    "doc_type",
    "doc_period",
    "pdf_filename",
    "pdf_path",
    "question_type",
    "question_reasoning",
    "justification",
    "evidence"
]

existing_priority_cols = [c for c in priority_cols if c in working_df.columns]
remaining_cols = [c for c in working_df.columns if c not in existing_priority_cols]

working_df = working_df[existing_priority_cols + remaining_cols]
working_df.head()

In [ ]:
working_csv_path = INTERIM_DIR / "financebench_open_source_working.csv"
working_parquet_path = INTERIM_DIR / "financebench_open_source_working.parquet"

working_df.to_csv(working_csv_path, index=False)
print("Saved CSV to:", working_csv_path)

try:
    working_df.to_parquet(working_parquet_path, index=False)
    print("Saved Parquet to:", working_parquet_path)
except ImportError as e:
    print("Parquet save skipped.")
    print("Reason:", e)
    print("Install pyarrow with: pip install pyarrow")

In [ ]:
unmatched_path = INTERIM_DIR / "financebench_unmatched_docs.csv"
unmatched_docs.to_csv(unmatched_path, index=False)

print("Saved unmatched docs to:", unmatched_path)

In [ ]:
output_path = DATA_DIR / "interim" / "financebench_sample_working.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

working_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

In [ ]:
unmatched_path = INTERIM_DIR / "financebench_unmatched_docs.csv"
unmatched_docs.to_csv(unmatched_path, index=False)

print("Saved unmatched docs to:", unmatched_path)

## Συμπέρασμα

Σε αυτό το notebook:

- φορτώθηκαν τα δύο JSONL αρχεία του FinanceBench sample
- ενώθηκαν questions και document metadata με βάση το `doc_name`
- ελέγχθηκαν τα τοπικά PDFs
- πραγματοποιήθηκε πρώτη αντιστοίχιση dataset documents ↔ local files
- αποθηκεύτηκε working dataset για τα επόμενα στάδια

Το επόμενο notebook είναι το `03_parse_pdfs_with_docling.ipynb`.